# Topics of the 1914 Press

What was *L'Humanité* writing about in 1914, and how did the news turn that summer? Without a single label, BERTopic groups similar passages and describes each group by its most characteristic words.

1. Find the topics
2. Tune them
3. Follow them over time

The passages' *embeddings* are precomputed (with Qwen3-Embedding-0.6B, truncated to 512 dimensions): this notebook needs no GPU.

## Data

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/french-press-1914-passages/passages.parquet
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/french-press-1914-passages/embeddings-512.npy
%pip install -q "bertopic<1"

In [ ]:
import numpy as np
import pandas as pd

passages = pd.read_parquet("passages.parquet")
embeddings = np.load("embeddings-512.npy").astype(np.float32)

# L'Humanité only (Le Figaro has only 24 issues), and a sample of 20,000 passages so
# that the computation takes a few minutes.
humanite = passages[passages.newspaper == "L'Humanité"].sample(20_000, random_state=0)
docs = humanite.text.tolist()
dates = humanite.date.tolist()
vectors = embeddings[humanite.index]
print(len(docs), vectors.shape)

## 1. Find the Topics

Create a `BERTopic` model that reduces the embeddings to 5 dimensions with UMAP (reproducibly), describes topics with a `CountVectorizer` without stop words (to **describe** topics, stop words get in the way: the only use in this course where we remove them; the corpus is French, so are its stop words), and only keeps topics of at least 50 passages.

Fit it with `fit_transform` (on the precomputed vectors), then display `get_topic_info()`: how many topics? What share of the passages has no topic (topic $-1$)? Read the keywords of the 15 largest topics, and two representative passages (`get_representative_docs`) of three of them.

**Useful documentation**

- [`BERTopic`](https://maartengr.github.io/BERTopic/api/bertopic.html)
- [`BERTopic.fit_transform`](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.fit_transform)
- [`BERTopic.get_topic_info`](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.get_topic_info)
- [`BERTopic.get_representative_docs`](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.get_representative_docs)
- [UMAP's parameters](https://umap-learn.readthedocs.io/en/latest/parameters.html)
- [`CountVectorizer`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html)

In [ ]:
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from spacy.lang.fr.stop_words import STOP_WORDS
from umap import UMAP

STOP_WORDS = sorted(STOP_WORDS | {"qu", "mm", "mme"})

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=0)`: a reduction to 5 dimensions, reproducible
- `CountVectorizer(stop_words=STOP_WORDS, min_df=10)`: `min_df=10` ignores the words found in fewer than 10 passages
- `BERTopic(language="french", umap_model=..., vectorizer_model=..., min_topic_size=50)`: without `language="french"`, BERTopic strips every accented character from the keywords ("grève" becomes "grve")
- `fit_transform(docs, vectors)`: the second argument avoids computing the embeddings again; returns each passage's topic, then probabilities
- `get_topic_info()`: one row per topic, topic −1 being the passages with no topic; `np.mean(np.array(topics) == -1)`: their share
- `get_topic(topic)`: a topic's keywords and their weights; `get_representative_docs(topic)`: its representative passages

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
topic_model = BERTopic(
  language="french",
  umap_model=UMAP(
    n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=0
  ),
  vectorizer_model=CountVectorizer(stop_words=STOP_WORDS, min_df=10),
  min_topic_size=...,
)
topics, _ = topic_model.fit_transform(
  ..., ...
)  # the texts, then the precomputed vectors
info = topic_model.get_topic_info()
print(
  len(info) - 1, "topics,", f"{np.mean(np.array(topics) == -1):.0%} without a topic"
)
info.head(16)

for topic in info.Topic[1:4]:
  print(topic, topic_model.get_topic(topic)[:8])
  for doc in topic_model.get_representative_docs(...)[:2]:
    print("   ", doc[:200])
```

### Solution

In [ ]:
topic_model = BERTopic(
  language="french",
  umap_model=UMAP(
    n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=0
  ),
  vectorizer_model=CountVectorizer(stop_words=STOP_WORDS, min_df=10),
  min_topic_size=50,
)
topics, _ = topic_model.fit_transform(docs, vectors)
info = topic_model.get_topic_info()
print(
  len(info) - 1, "topics,", f"{np.mean(np.array(topics) == -1):.0%} without a topic"
)
info.head(16)

In [ ]:
for topic in info.Topic[1:4]:
  print(topic, topic_model.get_topic(topic)[:8])
  for doc in topic_model.get_representative_docs(topic)[:2]:
    print("   ", doc[:200])

## 2. Tune the Topics

- Rerun with `min_topic_size=20`, then `200`: how many topics, what share without a topic?
- With the first model, reassign the passages without a topic to their nearest topic (`reduce_outliers`), then update the descriptions (`update_topics`).
- Find two topics about the same thing and merge them (`merge_topics`).

**Useful documentation**

- [Outlier reduction](https://maartengr.github.io/BERTopic/getting_started/outlier_reduction/outlier_reduction.html)
- [`BERTopic.update_topics`](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.update_topics)
- [`BERTopic.merge_topics`](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.merge_topics)

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- rerunning with another size: in a loop over `(20, 200)`, a new `BERTopic` with the same configuration, with `min_topic_size=size` (without overwriting `topic_model`, the first model)
- `topic_model.reduce_outliers(docs, topics, strategy="embeddings", embeddings=vectors)` returns the passages' new topics
- `topic_model.update_topics(docs, topics=new_topics, vectorizer_model=CountVectorizer(stop_words=STOP_WORDS, min_df=10))`: recomputes the keywords after the reassignment
- `topic_model.merge_topics(docs, [a, b])`: `a` and `b` are two topics to merge into one, picked by reading the keywords of `get_topic_info()`. Topic numbers change after a merge: `merge_topics` renumbers every topic and returns nothing (`None`), so take `topics = topic_model.topics_` again right after it

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
for size in (20, 200):
  model = BERTopic(
    language="french",
    umap_model=UMAP(
      n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=0
    ),
    vectorizer_model=CountVectorizer(stop_words=STOP_WORDS, min_df=10),
    min_topic_size=...,
  )
  found, _ = model.fit_transform(docs, vectors)
  print(
    f"min_topic_size={size}: {len(set(found)) - 1} topics, {np.mean(np.array(found) == -1):.0%} without a topic"
  )

new_topics = topic_model.reduce_outliers(docs, topics, strategy=..., embeddings=vectors)
topic_model.update_topics(docs, topics=new_topics, vectorizer_model=...)
print(f"{np.mean(np.array(new_topics) == -1):.0%} without a topic after reassignment")
topics = new_topics

topic_model.merge_topics(docs, [..., ...])  # two topics about the same thing
topics = topic_model.topics_  # the numbers have changed
```

### Solution

In [ ]:
for size in (20, 200):
  model = BERTopic(
    language="french",
    umap_model=UMAP(
      n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=0
    ),
    vectorizer_model=CountVectorizer(stop_words=STOP_WORDS, min_df=10),
    min_topic_size=size,
  )
  found, _ = model.fit_transform(docs, vectors)
  print(
    f"min_topic_size={size}: {len(set(found)) - 1} topics, {np.mean(np.array(found) == -1):.0%} without a topic"
  )

In [ ]:
new_topics = topic_model.reduce_outliers(
  docs, topics, strategy="embeddings", embeddings=vectors
)
topic_model.update_topics(
  docs,
  topics=new_topics,
  vectorizer_model=CountVectorizer(stop_words=STOP_WORDS, min_df=10),
)
print(f"{np.mean(np.array(new_topics) == -1):.0%} without a topic after reassignment")
topics = new_topics

# Here, two topics are about the legislative elections of April-May 1914: the candidates
# on one side, the abbreviated result tables on the other. Their numbers change with the
# libraries' versions: we find them through their shared keyword, "voix" (votes).
elections = [
  topic
  for topic in topic_model.get_topics()
  if "voix" in [word for word, _ in topic_model.get_topic(topic)]
]
for topic in elections:
  print(topic, [word for word, _ in topic_model.get_topic(topic)[:8]])

topic_model.merge_topics(docs, elections)
topics = topic_model.topics_  # the merge renumbered the topics

## 3. Follow the Topics over Time

`topic_model.topics_over_time` counts each topic's passages per week; `topic_model.visualize_topics_over_time` plots them. Show the topics about the war (*guerre*), Jaurès and the Caillaux trial: write a function `topic_with(word)` that returns the topic with the largest share of passages containing `word`. What happens between July and August 1914?

**Useful documentation**

- [Topics over time](https://maartengr.github.io/BERTopic/getting_started/topicsovertime/topicsovertime.html)
- [`BERTopic.topics_over_time`](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.topics_over_time)
- [`BERTopic.visualize_topics_over_time`](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.visualize_topics_over_time)
- [`Series.idxmax`](https://pandas.pydata.org/docs/reference/api/pandas.Series.idxmax.html)

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `topic_model.topics_over_time(docs, dates, nr_bins=52)`: 52 bins, a week each over a year
- `topic_model.visualize_topics_over_time(over_time, topics=chosen)`: `topics=` is the list of topics to plot
- `topic_with(word)`: among the passages containing the word (`word in doc.lower()`), the number of passages of each topic (`pd.Series(topics)[mask].value_counts()`), divided by that topic's total size (`pd.Series(topics).value_counts()`), without topic −1 (`.drop(-1, errors="ignore")`)
- `Series.idxmax()`: the label of the largest value, here the topic's number; try `"guerre"`, `"jaurès"` and `"caillaux"`

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
over_time = topic_model.topics_over_time(docs, dates, nr_bins=...)


def topic_with(word: str) -> int:
  mentions = pd.Series(topics)[[word in doc.lower() for doc in docs]].value_counts()
  share = (mentions / pd.Series(topics).value_counts()).drop(-1, errors="ignore")
  return int(share.idxmax())


chosen = [topic_with(word) for word in ("guerre", "jaurès", ...)]
print(chosen)
topic_model.visualize_topics_over_time(over_time, topics=...)
```

### Solution

In [ ]:
over_time = topic_model.topics_over_time(docs, dates, nr_bins=52)


def topic_with(word: str) -> int:
  mentions = pd.Series(topics)[[word in doc.lower() for doc in docs]].value_counts()
  share = (mentions / pd.Series(topics).value_counts()).drop(-1, errors="ignore")
  return int(share.idxmax())


chosen = [topic_with(word) for word in ("guerre", "jaurès", "caillaux")]
print(chosen)
topic_model.visualize_topics_over_time(over_time, topics=chosen)

# In March, Mme Caillaux kills Gaston Calmette, the editor of Le Figaro; her trial fills
# July. Jaurès is assassinated on July 31. From August on, the war and the reports from the
# front dominate.